# 01 — Data Extraction & Profiling

This is the first step of our NYC Airbnb analysis. We're using the **Inside Airbnb** dataset for New York City (2019).

**Goal:** Load the raw data, see what's inside, and figure out what needs cleaning.

### Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Settings for better display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

# Path to our data
data_path = Path('../data/raw/AB_NYC_2019.csv')
if not data_path.exists():
    data_path = Path('data/raw/AB_NYC_2019.csv') # fallback if run from root

### Loading the data

In [ ]:
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
df.head()

### Initial Data Profiling
Let's check data types and look for missing values.

In [ ]:
df.info()

In [ ]:
# Checking for nulls
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary[null_summary['null_count'] > 0].sort_values('null_count', ascending=False)

**Observations:**
- `reviews_per_month` and `last_review` have a lot of missing values (~20%). This probably means these listings haven't been reviewed yet.
- `name` and `host_name` have a few missing entries, but not many.

### Exploring Main Categories

In [ ]:
print("Neighbourhood Groups:")
print(df['neighbourhood_group'].value_counts())

print("\nRoom Types:")
print(df['room_type'].value_counts())

### Checking Numeric Ranges
We need to see if there are any weird values like $0 prices or extreme minimum nights.

In [ ]:
df[['price', 'minimum_nights', 'number_of_reviews', 'availability_365']].describe()

In [ ]:
# Let's look closer at the anomalies
print(f"Listings with price 0: {len(df[df['price'] == 0])}")
print(f"Listings with min nights > 365: {len(df[df['minimum_nights'] > 365])}")
print(f"Listings with 0 availability: {len(df[df['availability_365'] == 0])}")

### Summary of Findings
Here's what we found that needs fixing in the next notebook:
1. **Missing Values**: Fill `reviews_per_month` with 0 and `name`/`host_name` with placeholders.
2. **Zero Prices**: 11 listings have a price of $0. We should remove these.
3. **Price Outliers**: Max price is $10,000, which is likely an outlier. We'll need to cap this.
4. **Minimum Nights**: Some values are over 365 days, which seems like a mistake for a rental platform.